In [ ]:
import os
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
from torchvision import transforms
import torch
import torch.nn as nn

In [ ]:
import matplotlib.pyplot as plt

with open('AFAC/annotations/train.txt', 'r') as f:
    ages = []
    for line in f:
        try:
            _, age_str = line.strip().split()
            ages.append(int(age_str))
        except:
            continue

plt.hist(ages, bins=range(0, max(ages)+12, 12), color='salmon', edgecolor='black')
plt.xlabel("Age (months)")
plt.ylabel("Number of images")
plt.title("AFAC Age Distribution")
plt.grid(axis='y')
plt.xticks(range(0, max(ages)+12, 12))
plt.show()

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

txt_path = "AFAC/annotations/train.txt"

def age_to_class(age_months):
    if age_months <= 48:
        return "Young"
    elif age_months <= 83:
        return "Middle"
    else:
        return "Senior"

age_classes = []
age_raw = []

with open(txt_path, 'r') as f:
    for line in f:
        try:
            _, age_str = line.strip().split()
            age = int(age_str)
            age_raw.append(age)
            age_classes.append(age_to_class(age))
        except:
            print(f"⚠️ 잘못된 라인 무시: {line.strip()}")

class_counts = Counter(age_classes)
print("✅ 클래스 분포 (3-Class 기준):")
for cls, count in class_counts.items():
    print(f"{cls}: {count}개")

plt.figure(figsize=(10, 4))
plt.hist(age_raw, bins=40, color='salmon', edgecolor='black')
plt.title("AFAC Age Distribution (in Months)")
plt.xlabel("Age (months)")
plt.ylabel("Number of images")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(range(0, max(age_raw)+12, 12))
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
plt.bar(class_counts.keys(), class_counts.values(), color='skyblue')
plt.title("AFAC Age Class Distribution (3-Class)")
plt.xlabel("Class")
plt.ylabel("Count")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Use dogs up to 48 months as Young and dogs from 84 months as Senior; exclude the ambiguous middle range.

# ================================
# 🐶 AFAC Dataset - Age-based Class Reassignment (2-Class)
# ================================







In [ ]:
# Convert age annotations into two classes and keep records with available images.

import os
from collections import Counter

train_txt_path = "AFAC/annotations/train.txt"

image_root = "AFAC/trainset"

save_txt_path = "AFAC/AFAC.txt"

def age_to_class(age_months):
    if age_months <= 48:
        return "Young"
    elif age_months >= 84:
        return "Senior"
    else:
        return "Middle"

records = []
class_counts = Counter()

with open(train_txt_path, "r") as f:
    for line in f:
        try:
            filename, age_str = line.strip().split()
            age = int(age_str)
            label = age_to_class(age)

            if label == "Middle":
                continue

            image_path = os.path.join(image_root, filename)
            if not os.path.isfile(image_path):
                print(f"⚠️ 이미지 누락: {filename}")
                continue

            records.append((filename, label))
            class_counts[label] += 1
        except:
            print(f"⚠️ 잘못된 라인 무시: {line.strip()}")

with open(save_txt_path, "w") as f:
    for filename, label in records:
        f.write(f"{filename}\t{label}\n")

print(f"\n✅ {len(records)}개 샘플이 {save_txt_path}에 저장되었습니다.")
print("\n📊 클래스 분포 (Middle 제거 후):")
for cls in ["Young", "Senior"]:
    print(f"{cls}: {class_counts[cls]}개")

In [ ]:
# Resize and augment images before training.

import os
from PIL import Image
from tqdm import tqdm

import torch
from torchvision import transforms

input_dir = 'AFAC/trainset'
output_dir = 'AFAC/preprocessed'
os.makedirs(output_dir, exist_ok=True)

preprocess = transforms.Compose([
    transforms.Resize((256, 256)),

    transforms.Lambda(lambda img: img.crop((
        16,    # left
        12,    # top
        240,   # right = left + 224
        236    # bottom = top + 224
    ))),

    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.05,
        saturation=0.1,
        hue=0.02
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ToTensor()
])

image_paths = [f for f in os.listdir(input_dir) if f.lower().endswith(('.jpg', '.png'))]

for filename in tqdm(image_paths, desc="📦 Preprocessing AFAC"):
    try:
        img = Image.open(os.path.join(input_dir, filename)).convert('RGB')
        processed = preprocess(img)
        output_path = os.path.join(output_dir, filename.replace(".png", ".jpg"))
        transforms.ToPILImage()(processed).save(output_path)
    except Exception as e:
        print(f"❌ {filename} 처리 실패: {e}")

In [ ]:
len([f for f in os.listdir("AFAC/trainset") if f.lower().endswith((".jpg", ".jpeg", ".png"))]) 

In [ ]:
import torch
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, z_dim=100, c_dim=2):
        super().__init__()
        self.fc = nn.Linear(z_dim + c_dim, 1024 * 4 * 4)

        self.net = nn.Sequential(
            nn.BatchNorm2d(1024),
            nn.ReLU(True),

            # 4×4 → 8×8
            nn.ConvTranspose2d(1024, 512, 4, 2, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            # 8×8 → 16×16
            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            # 16×16 → 32×32
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            # 32×32 → 64×64
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            # 64×64 → 128×128
            nn.ConvTranspose2d(64, 3, 4, 2, 1),
            nn.Tanh()  # Output in [-1, 1]
        )

    def forward(self, z, c):
        x = torch.cat([z, c], dim=1)         # [B, z_dim + c_dim]
        x = self.fc(x)                       # [B, 1024 * 4 * 4]
        x = x.view(-1, 1024, 4, 4)           # [B, 1024, 4, 4]
        return self.net(x)                   # [B, 3, 128, 128]

In [ ]:
# Inject the age condition at selected generator stages.

import torch
import torch.nn as nn
import torch.nn.functional as F

class InjectionGenerator(nn.Module):
    def __init__(self, z_dim=128, c_dim=2):
        super().__init__()
        self.fc = nn.Linear(z_dim + c_dim, 1024 * 4 * 4)

        self.inject_c1 = nn.Linear(c_dim, 512)
        self.inject_c2 = nn.Linear(c_dim, 256)

        # deconv blocks
        self.net = nn.Sequential(
            nn.BatchNorm2d(1024),
            nn.ReLU(True),

            nn.ConvTranspose2d(1024, 512, 4, 2, 1),   # 4×4 → 8×8
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.ConvTranspose2d(512, 256, 4, 2, 1),    # 8×8 → 16×16
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),    # 16×16 → 32×32
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),     # 32×32 → 64×64
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 3, 4, 2, 1),       # 64×64 → 128×128
            nn.Tanh()
        )

    def forward(self, z, c):
        x = torch.cat([z, c], dim=1)             # [B, z_dim + c_dim]
        x = self.fc(x).view(-1, 1024, 4, 4)      # [B, 1024, 4, 4]

        # Initial batchnorm + relu
        x = self.net[0](x)
        x = self.net[1](x)

        # Block 1: 4x4 → 8x8 + c inject
        x = self.net[2](x)
        x += self.inject_c1(c).view(-1, 512, 1, 1)
        x = self.net[3](x)
        x = self.net[4](x)

        # Block 2: 8x8 → 16x16 + c inject
        x = self.net[5](x)
        x += self.inject_c2(c).view(-1, 256, 1, 1)
        x = self.net[6](x)
        x = self.net[7](x)

        # Block 3: 16x16 → 32x32
        x = self.net[8](x)
        x = self.net[9](x)
        x = self.net[10](x)

        # Block 4: 32x32 → 64x64
        x = self.net[11](x)
        x = self.net[12](x)
        x = self.net[13](x)

        # Output: 64x64 → 128x128 → Tanh
        x = self.net[14](x)
        x = self.net[15](x)

        return x

In [ ]:
# Compare a generator with condition injection at one upsampling block.

import torch
import torch.nn as nn
import torch.nn.functional as F

class ConcatGenerator(nn.Module):
    def __init__(self, z_dim=128, c_dim=2, c_emb_dim=16):
        super().__init__()
        self.c_emb = nn.Linear(c_dim, c_emb_dim)             # 🔹 2 → 16
        self.fc = nn.Linear(z_dim + c_emb_dim, 1024 * 4 * 4)  # z + c_emb

        self.inject_c = nn.Linear(c_emb_dim, 256)

        # 🔹 deconv blocks
        self.net = nn.Sequential(
            nn.BatchNorm2d(1024),
            nn.ReLU(True),

            nn.ConvTranspose2d(1024, 512, 4, 2, 1),   # 4x4 → 8x8
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.ConvTranspose2d(512, 256, 4, 2, 1),    # 8x8 → 16x16
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),    # 16x16 → 32x32
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),     # 32x32 → 64x64
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 3, 4, 2, 1),       # 64x64 → 128x128
            nn.Tanh()
        )

    def forward(self, z, c):
        # 🔹 Embed condition
        c_emb = self.c_emb(c)  # [B, 16]

        # 🔹 Initial projection
        x = torch.cat([z, c_emb], dim=1)              # [B, z + c_emb]
        x = self.fc(x).view(-1, 1024, 4, 4)

        # 🔹 Block 1: 4x4 → 8x8
        x = self.net[0](x)
        x = self.net[1](x)

        x = self.net[2](x)
        x = self.net[3](x)
        x = self.net[4](x)

        # 🔹 Block 2: 8x8 → 16x16 + condition inject
        x = self.net[5](x)
        x += self.inject_c(c_emb).view(-1, 256, 1, 1)
        x = self.net[6](x)
        x = self.net[7](x)

        # 🔹 Block 3: 16x16 → 32x32
        x = self.net[8](x)
        x = self.net[9](x)
        x = self.net[10](x)

        # 🔹 Block 4: 32x32 → 64x64
        x = self.net[11](x)
        x = self.net[12](x)
        x = self.net[13](x)

        # 🔹 Output: 64x64 → 128x128
        x = self.net[14](x)
        x = self.net[15](x)

        return x

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import spectral_norm as SN

class ProjectionDiscriminator(nn.Module):
    def __init__(self, c_dim=2, img_channels=3):
        super().__init__()
        self.c_dim = c_dim

        self.feature_extractor = nn.Sequential(
            SN(nn.Conv2d(img_channels, 64, 4, 2, 1)),   # 128 → 64
            nn.LeakyReLU(0.2, inplace=True),

            SN(nn.Conv2d(64, 128, 4, 2, 1)),             # 64 → 32
            nn.LeakyReLU(0.2, inplace=True),

            SN(nn.Conv2d(128, 256, 4, 2, 1)),            # 32 → 16
            nn.LeakyReLU(0.2, inplace=True),

            SN(nn.Conv2d(256, 512, 4, 2, 1)),            # 16 → 8
            nn.LeakyReLU(0.2, inplace=True),

            SN(nn.Conv2d(512, 512, 4, 2, 1)),            # 8 → 4
            nn.LeakyReLU(0.2, inplace=True),

            SN(nn.Conv2d(512, 512, 4, 2, 1)),            # 4 → 2
            nn.LeakyReLU(0.2, inplace=True),

            SN(nn.Conv2d(512, 512, 2, 1, 0)),            # 2 → 1
        )

        self.linear = SN(nn.Linear(512, 1))              # unconditional logit
        self.embed = SN(nn.Embedding(c_dim, 512))        # class embedding (same dim as features)

    def forward(self, x, c):
        """
        x: [B, 3, 128, 128]
        c: [B] (LongTensor class label)
        """
        features = self.feature_extractor(x)             # [B, 512, 1, 1]
        features = features.view(x.size(0), -1)          # [B, 512]

        out = self.linear(features)                      # [B, 1]
        emb = self.embed(c)                              # [B, 512]
        proj = torch.sum(features * emb, dim=1, keepdim=True)  # [B, 1]

        return out + proj                                # [B, 1]

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import spectral_norm as SN

class ConcatDiscriminator(nn.Module):
    def __init__(self, c_dim=2, img_channels=3, embed_dim=16):
        super().__init__()
        self.c_dim = c_dim

        # ⚙️ Feature extractor
        self.feature_extractor = nn.Sequential(
            SN(nn.Conv2d(img_channels, 64, 4, 2, 1)),   # 128 → 64
            nn.LeakyReLU(0.2, inplace=True),

            SN(nn.Conv2d(64, 128, 4, 2, 1)),             # 64 → 32
            nn.LeakyReLU(0.2, inplace=True),

            SN(nn.Conv2d(128, 256, 4, 2, 1)),            # 32 → 16
            nn.LeakyReLU(0.2, inplace=True),

            SN(nn.Conv2d(256, 512, 4, 2, 1)),            # 16 → 8
            nn.LeakyReLU(0.2, inplace=True),

            SN(nn.Conv2d(512, 512, 8, 1, 0)),            # 8×8 → 1×1
            nn.LeakyReLU(0.2, inplace=True),
        )

        # ⚙️ Class embedding
        self.embed = nn.Linear(c_dim, embed_dim)         # e.g. 2 → 16

        # ⚙️ Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(512 + embed_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1)
        )

    def forward(self, x, c):
        """
        x: [B, 3, 128, 128]
        c: [B, c_dim] (e.g. one-hot)
        """
        B = x.size(0)
        features = self.feature_extractor(x)             # [B, 512, 1, 1]
        features = features.view(B, -1)                  # [B, 512]

        c_emb = self.embed(c)                            # [B, embed_dim]
        concat = torch.cat([features, c_emb], dim=1)     # [B, 512 + embed_dim]

        out = self.classifier(concat)                    # [B, 1]
        return out

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Discriminator(nn.Module):
    def __init__(self, c_dim=2, img_channels=3):
        super().__init__()
        self.c_dim = c_dim  # for one-hot

        self.net = nn.Sequential(
            # Input: [B, 3 + c_dim, 128, 128]
            nn.Conv2d(img_channels + c_dim, 64, 4, 2, 1),   # 64×64
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1),                    # 32×32
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1),                   # 16×16
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, 4, 2, 1),                   # 8×8
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 1, 8),                           # 1×1
        )

    def forward(self, x, c):
        """
        x: [B, 3, 128, 128]
        c: [B] (LongTensor class label)
        """
        B, _, H, W = x.shape
        c_onehot = F.one_hot(c, num_classes=self.c_dim).float()  # [B, c_dim]
        c_map = c_onehot.view(B, self.c_dim, 1, 1).expand(B, self.c_dim, H, W)  # [B, c_dim, H, W]
        x_cond = torch.cat([x, c_map], dim=1)  # [B, 3 + c_dim, H, W]
        out = self.net(x_cond)                 # [B, 1, 1, 1]
        return out.view(-1, 1)                 # [B, 1]

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class AFACDataset(Dataset):
    def __init__(self, txt_path, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform

        self.samples = []
        self.label_map = {'Young': 0, 'Senior': 1}
        self.missing_files = []

        with open(txt_path, 'r') as f:
            for line in f:
                filename, label = line.strip().split('\t')

                img_path = os.path.join(img_dir, filename)
                if os.path.exists(img_path):
                    self.samples.append((filename, label))
                else:
                    self.missing_files.append(filename)

        print(f"✅ 총 로드된 샘플 수: {len(self.samples)}")
        if self.missing_files:
            print(f"⚠️ 누락된 이미지 파일 수: {len(self.missing_files)}")
            print(f"예시 누락 파일명: {self.missing_files[:5]}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        filename, label = self.samples[idx]
        img_path = os.path.join(self.img_dir, filename)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label_idx = self.label_map[label]
        return image, label_idx

In [ ]:
txt_path = "AFAC/AFAC.txt"
img_dir = "AFAC/preprocessed"

dataset = AFACDataset(txt_path, img_dir, transform=None)

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt
from torchvision.transforms import ToPILImage
from torchvision import transforms
from tqdm import tqdm

txt_path = 'AFAC/AFAC.txt'
raw_dir = 'AFAC/trainset'
pre_dir = 'AFAC/preprocessed'

to_pil = ToPILImage()

label_map = {'Young': 0, 'Senior': 1}

samples = []
with open(txt_path, 'r') as f:
    for line in f:
        filename, label = line.strip().split('\t')
        samples.append((filename, label_map[label]))

for i in range(5):
    filename, label = samples[i]

    raw_img = Image.open(os.path.join(raw_dir, filename)).convert('RGB')

    pre_img_path = os.path.join(pre_dir, filename.replace('.png', '.jpg'))
    pre_img = Image.open(pre_img_path).convert('RGB')

    plt.figure(figsize=(8, 4))
    plt.suptitle(f"Label: {'Young' if label == 0 else 'Senior'} (class={label})", fontsize=14)

    plt.subplot(1, 2, 1)
    plt.imshow(raw_img)
    plt.title('Before Preprocess')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(pre_img)
    plt.title('After Preprocess')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:

# ------------------------------
# ------------------------------
batch_size = 64
# z_dim = 100
z_dim = 128
c_dim = 2
lr = 2e-4
num_epochs = 100

device = 'cuda' if torch.cuda.is_available() else 'cpu'

from torch.utils.data import DataLoader

# ------------------------------
# 2. Transform
# ------------------------------
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,) * 3, (0.5,) * 3)
])

# ------------------------------
# ------------------------------
dataset = AFACDataset("AFAC/AFAC.txt", "AFAC/preprocessed", transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4)

import matplotlib.pyplot as plt
import torchvision.utils as vutils

real_imgs, real_labels = next(iter(dataloader))

def denormalize(t):
    return t * 0.5 + 0.5

grid = vutils.make_grid(denormalize(real_imgs[:8]), nrow=4)
plt.figure(figsize=(8, 4))
plt.imshow(grid.permute(1, 2, 0))  # (C, H, W) → (H, W, C)
plt.title("💡 Real Images from Dataloader")
plt.axis('off')
plt.show()

print("Labels:", real_labels[:8].tolist())

In [ ]:
# Train the age-conditional generator and discriminator.

import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, utils
import matplotlib.pyplot as plt
from tqdm import tqdm
from torchvision.utils import save_image
from torch.optim.lr_scheduler import StepLR

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1 or classname.find('ConvTranspose') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# ------------------------------
# ------------------------------
batch_size = 64
# z_dim = 100
z_dim = 128
c_dim = 2
lr = 2e-4
num_epochs = 300

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ------------------------------
# 2. Transform
# ------------------------------
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,) * 3, (0.5,) * 3)
])

# ------------------------------
# ------------------------------
dataset = AFACDataset("AFAC/AFAC.txt", "AFAC/preprocessed", transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4)

# ------------------------------
# ------------------------------
# G = Generator(z_dim=z_dim, c_dim=c_dim).to(device)
G = InjectionGenerator(z_dim=z_dim, c_dim=c_dim).to(device)
# D = Discriminator(c_dim=c_dim).to(device)
D = ProjectionDiscriminator(c_dim=c_dim).to(device)


G.apply(weights_init)
D.apply(weights_init)

# ------------------------------
# ------------------------------
# bce_loss = nn.BCEWithLogitsLoss()
optimizer_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
scheduler_G = StepLR(optimizer_G, step_size=50, gamma=0.5)
optimizer_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

# ------------------------------
# ------------------------------        
# def generate_labels(size, value):
#     return torch.full((size, 1), value, dtype=torch.float, device=device)

def denormalize(t):
    return t * 0.5 + 0.5

def show_samples(generated_imgs, labels, title):
    grid = utils.make_grid(denormalize(generated_imgs[:8].cpu()), nrow=4)
    plt.figure(figsize=(8, 4))
    plt.imshow(grid.permute(1, 2, 0))
    plt.title(title)
    plt.axis('off')
    plt.show()

# ------------------------------
# ------------------------------
best_g_loss = float('inf')
z_fixed = torch.randn(20, z_dim, device=device)

for epoch in range(num_epochs):
    G.train()
    D.train()
    epoch_start = time.time()
    loop = tqdm(dataloader, desc=f"[Epoch {epoch + 1}/{num_epochs}]", leave=False)

    for real_imgs, real_labels in loop:
        real_imgs = real_imgs.to(device)
        real_labels = real_labels.to(device)
        batch_size = real_imgs.size(0)

        z = torch.randn(batch_size, z_dim, device=device)

        # ▒ Train D ▒
        D.zero_grad()
        
        real_out = D(real_imgs, real_labels)  # [B, 1]
        loss_D_real = torch.mean(F.relu(1.0 - real_out))  # hinge loss for real
        
        fake_labels = torch.randint(0, c_dim, (batch_size,), device=device)
        fake_cond = F.one_hot(fake_labels, num_classes=c_dim).float()
        fake_imgs = G(z, fake_cond)
        
        fake_out = D(fake_imgs.detach(), fake_labels)
        loss_D_fake = torch.mean(F.relu(1.0 + fake_out))  # hinge loss for fake
        
        d_loss = loss_D_real + loss_D_fake
        d_loss.backward()
        optimizer_D.step()
        
        # ▒ Train G ▒
        G.zero_grad()
        fake_imgs = G(z, fake_cond)
        fake_out = D(fake_imgs, fake_labels)
        
        g_loss = -torch.mean(fake_out)  # generator tries to increase D output
        
        g_loss.backward()
        optimizer_G.step()

        loop.set_postfix(d_loss=d_loss.item(), g_loss=g_loss.item())

    scheduler_G.step()
    elapsed = time.time() - epoch_start
    print(f"✅ Epoch {epoch + 1:03d} 완료 | D_loss: {d_loss.item():.4f} | G_loss: {g_loss.item():.4f} | 소요시간: {elapsed / 60:.2f}분")

    # ------------------------------
    # ------------------------------
    save_every = 10
    version = "injected"
    
    ckpt_dir = os.path.join("checkpoints", version)
    sample_dir = os.path.join("samples", version)
    
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(sample_dir, exist_ok=True)
    
    if (epoch + 1) % save_every == 0:
        ckpt_path = os.path.join(ckpt_dir, f"G_epoch_{epoch+1:03d}.pth")
        checkpoint = {
            'epoch': epoch + 1,
            'G_state_dict': G.state_dict(),
            'optimizer_G_state_dict': optimizer_G.state_dict(),
            'g_loss': g_loss.item(),
            'z_sample': z[:20].detach().cpu(),
            'c_sample': fake_labels[:20].detach().cpu()
        }
        torch.save(checkpoint, ckpt_path)
        
        img_path = os.path.join(sample_dir, f"sample_epoch_{epoch+1:03d}.png")
        
        with torch.no_grad():
            z_fixed = z_fixed
    
            c_young = torch.zeros(20, dtype=torch.long, device=device)
            c_senior = torch.ones(20, dtype=torch.long, device=device)
    
            cond_young = F.one_hot(c_young, num_classes=c_dim).float()
            cond_senior = F.one_hot(c_senior, num_classes=c_dim).float()
    
            gen_young = G(z_fixed, cond_young)   # [20, 3, 128, 128]
            gen_senior = G(z_fixed, cond_senior) # [20, 3, 128, 128]
    
            interleaved = torch.stack([gen_young, gen_senior], dim=1)  # [20, 2, 3, 128, 128]
            sample_imgs = interleaved.flatten(0, 1)                    # [40, 3, 128, 128]
            save_image(denormalize(sample_imgs), img_path, nrow=10)

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, utils
import matplotlib.pyplot as plt
from torchvision.utils import save_image
from torch.optim.lr_scheduler import StepLR
from tqdm import tqdm

# ------------------------------
# ------------------------------
batch_size = 64
z_dim = 128
c_dim = 2
lr = 2e-4
num_epochs = 300
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ------------------------------
# 2. Transform
# ------------------------------
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,) * 3, (0.5,) * 3)
])

# ------------------------------
# ------------------------------
dataset = AFACDataset("AFAC/AFAC.txt", "AFAC/preprocessed", transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4)

# ------------------------------
# ------------------------------
G = ConcatGenerator(z_dim=z_dim, c_dim=c_dim, c_emb_dim=16).to(device)
D = ConcatDiscriminator(c_dim=c_dim, embed_dim=16).to(device)

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1 or classname.find('ConvTranspose') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

G.apply(weights_init)
D.apply(weights_init)

# ------------------------------
# ------------------------------
optimizer_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
scheduler_G = StepLR(optimizer_G, step_size=50, gamma=0.5)
optimizer_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

# ------------------------------
# ------------------------------
def denormalize(t):
    return t * 0.5 + 0.5

# ------------------------------
# ------------------------------
best_g_loss = float('inf')
z_fixed = torch.randn(20, z_dim, device=device)

for epoch in range(num_epochs):
    G.train()
    D.train()
    epoch_start = time.time()
    loop = tqdm(dataloader, desc=f"[Epoch {epoch + 1}/{num_epochs}]", leave=False)

    for real_imgs, real_labels in loop:
        real_imgs = real_imgs.to(device)
        real_labels = real_labels.to(device)
        batch_size = real_imgs.size(0)

        z = torch.randn(batch_size, z_dim, device=device)

        # ▒ Train D ▒
        D.zero_grad()
        real_out = D(real_imgs, F.one_hot(real_labels, num_classes=c_dim).float())
        loss_D_real = torch.mean(F.relu(1.0 - real_out))

        fake_labels = torch.randint(0, c_dim, (batch_size,), device=device)
        fake_cond = F.one_hot(fake_labels, num_classes=c_dim).float()
        fake_imgs = G(z, fake_cond)

        fake_out = D(fake_imgs.detach(), fake_cond)
        loss_D_fake = torch.mean(F.relu(1.0 + fake_out))

        d_loss = loss_D_real + loss_D_fake
        d_loss.backward()
        optimizer_D.step()

        # ▒ Train G ▒
        G.zero_grad()
        fake_imgs = G(z, fake_cond)
        fake_out = D(fake_imgs, fake_cond)
        g_loss = -torch.mean(fake_out)

        g_loss.backward()
        optimizer_G.step()

        loop.set_postfix(d_loss=d_loss.item(), g_loss=g_loss.item())

    scheduler_G.step()
    elapsed = time.time() - epoch_start
    print(f"✅ Epoch {epoch + 1:03d} 완료 | D_loss: {d_loss.item():.4f} | G_loss: {g_loss.item():.4f} | 소요시간: {elapsed / 60:.2f}분")

    # ------------------------------
    # ------------------------------
    save_every = 10
    version = "concat"
    ckpt_dir = os.path.join("checkpoints", version)
    sample_dir = os.path.join("samples", version)
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(sample_dir, exist_ok=True)

    if (epoch + 1) % save_every == 0:
        ckpt_path = os.path.join(ckpt_dir, f"G_epoch_{epoch+1:03d}.pth")
        checkpoint = {
            'epoch': epoch + 1,
            'G_state_dict': G.state_dict(),
            'optimizer_G_state_dict': optimizer_G.state_dict(),
            'g_loss': g_loss.item(),
            'z_sample': z_fixed[:20].detach().cpu(),
            'c_sample': fake_labels[:20].detach().cpu()
        }
        torch.save(checkpoint, ckpt_path)

        img_path = os.path.join(sample_dir, f"sample_epoch_{epoch+1:03d}.png")
        with torch.no_grad():
            c_young = torch.zeros(20, dtype=torch.long, device=device)
            c_senior = torch.ones(20, dtype=torch.long, device=device)

            cond_young = F.one_hot(c_young, num_classes=c_dim).float()
            cond_senior = F.one_hot(c_senior, num_classes=c_dim).float()

            gen_young = G(z_fixed, cond_young)
            gen_senior = G(z_fixed, cond_senior)

            interleaved = torch.stack([gen_young, gen_senior], dim=1)
            sample_imgs = interleaved.flatten(0, 1)
            save_image(denormalize(sample_imgs), img_path, nrow=10)

In [ ]:
# Compare generated samples from the candidate generators.

import os
import torch
import torch.nn.functional as F
from torchvision.utils import save_image
from torchvision import transforms


# ------------------------------
# ------------------------------
z_dim = 128
c_dim = 2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
epochs = list(range(100, 301, 10))

# ------------------------------
# ------------------------------
torch.manual_seed(42)

z_fixed_base = torch.randn(30, 100, device=device)
z_fixed_inj  = torch.randn(30, 128, device=device)

c_young = torch.zeros(30, dtype=torch.long, device=device)
c_senior = torch.ones(30, dtype=torch.long, device=device)
cond_y = F.one_hot(c_young, num_classes=c_dim).float()
cond_s = F.one_hot(c_senior, num_classes=c_dim).float()

# ------------------------------
# ------------------------------
def load_generator(checkpoint_path, model_type):
    if model_type == 'baseline':
        model = Generator(z_dim=100, c_dim=2).to(device)
    elif model_type == 'injected':
        model = InjectionGenerator(z_dim=128, c_dim=2).to(device)
    else:
        raise ValueError("model_type은 'baseline' 또는 'injected'이어야 합니다.")

    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state['G_state_dict'])
    model.eval()
    return model

# ------------------------------
# ------------------------------
output_dir = "samples/epoch_comparison"
os.makedirs(output_dir, exist_ok=True)

def denormalize(t):
    return t * 0.5 + 0.5

# ------------------------------
# 5. Loop over epochs
# ------------------------------
for epoch in epochs:
    print(f"🔄 Epoch {epoch} 처리 중...")

    base_ckpt = f"checkpoints/baseline/G_epoch_{epoch:03d}.pth"
    inj_ckpt = f"checkpoints/injected/G_epoch_{epoch:03d}.pth"

    G_base = load_generator(f"checkpoints/baseline/G_epoch_{epoch:03d}.pth", model_type='baseline')
    G_inj = load_generator(f"checkpoints/injected/G_epoch_{epoch:03d}.pth", model_type='injected')
  
    with torch.no_grad():
        base_y = G_base(z_fixed_base, cond_y)
        base_s = G_base(z_fixed_base, cond_s)
        inj_y  = G_inj(z_fixed_inj,  cond_y)
        inj_s  = G_inj(z_fixed_inj,  cond_s)

        all_imgs = torch.cat([base_y, base_s, inj_y, inj_s], dim=0)

        save_path = os.path.join(output_dir, f"compare_epoch_{epoch:03d}.png")
        save_image(denormalize(all_imgs), save_path, nrow=30)

    print(f"✅ 저장 완료: {save_path}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, z_dim=128, c_dim=2, img_channels=3):
        super().__init__()
        self.c_dim = c_dim
        self.z_dim = z_dim

        self.conv = nn.Sequential(
            nn.Conv2d(img_channels + c_dim, 64, 4, 2, 1),   # 128 → 64
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1),                    # 64 → 32
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1),                   # 32 → 16
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1),                   # 16 → 8
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 1024, 4, 2, 1),                  # 8 → 4
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),                        # 4×4 → 1×1
        )
        self.fc = nn.Linear(1024, z_dim)

    def forward(self, x, c):
        # c: one-hot vector [B, c_dim]
        B, _, H, W = x.shape
        c = c.view(B, self.c_dim, 1, 1).expand(-1, -1, H, W)
        x = torch.cat([x, c], dim=1)  # [B, 3 + c_dim, H, W]
        h = self.conv(x)              # [B, 1024, 1, 1]
        h = h.view(B, -1)             # [B, 1024]
        z = self.fc(h)                # [B, z_dim]
        return z

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConcatEncoder(nn.Module):
    def __init__(self, z_dim=128, c_dim=2, c_emb_dim=16, img_channels=3):
        super().__init__()
        self.c_emb = nn.Linear(c_dim, c_emb_dim)  # 🔹 c: one-hot → embed
        self.z_dim = z_dim

        # 🔹 CNN Feature Extractor
        self.conv = nn.Sequential(
            nn.Conv2d(img_channels, 64, 4, 2, 1),     # 128 → 64
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1),              # 64 → 32
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1),             # 32 → 16
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 512, 4, 2, 1),             # 16 → 8
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),

            nn.Conv2d(512, 512, 4, 2, 1),             # 8 → 4
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1)                   # 4x4 → 1x1
        )

        # 🔹 FC: concat([h, c_emb]) → z
        self.fc = nn.Sequential(
            nn.Linear(512 + c_emb_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, z_dim)
        )

    def forward(self, x, c):
        B = x.size(0)
        h = self.conv(x).view(B, -1)          # [B, 512]
        c_emb = self.c_emb(c)                # [B, 16]
        x_cat = torch.cat([h, c_emb], dim=1) # [B, 512+16]
        z = self.fc(x_cat)                   # [B, z_dim]
        return z

In [ ]:
# Train an encoder to initialize latent reconstruction.

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'

G = InjectionGenerator(z_dim=128, c_dim=2).to(device)
G.load_state_dict(torch.load("checkpoints/injected/G_epoch_250.pth")['G_state_dict'])
G.eval()

# E = Encoder(z_dim=128, c_dim=2).to(device)
E = ConcatEncoder(z_dim=128, c_dim=2, c_emb_dim=16).to(device)

optimizer = torch.optim.Adam(E.parameters(), lr=1e-4)
criterion = nn.MSELoss()

# ------------------------------
# ------------------------------
n_samples = 100_000
batch_size = 64
z_dim = 128
c_dim = 2

z_all = torch.randn(n_samples, z_dim)
c_labels_all = torch.randint(0, c_dim, (n_samples,))
c_onehot_all = F.one_hot(c_labels_all, num_classes=c_dim).float()

x_list, z_list, c_list = [], [], []

G.eval()
with torch.no_grad():
    for i in tqdm(range(0, n_samples, batch_size), desc="Generating synthetic data"):
        z_batch = z_all[i:i+batch_size].to(device)
        c_batch = c_onehot_all[i:i+batch_size].to(device)

        x_batch = G(z_batch, c_batch).cpu()
        x_list.append(x_batch)
        z_list.append(z_batch.cpu())
        c_list.append(c_batch.cpu())

x = torch.cat(x_list, dim=0)  # [n_samples, 3, 128, 128]
z = torch.cat(z_list, dim=0)  # [n_samples, z_dim]
c_onehot = torch.cat(c_list, dim=0)  # [n_samples, c_dim]

print(f"✅ Synthetic data 생성 완료: x.shape={x.shape}, z.shape={z.shape}, c.shape={c_onehot.shape}")

dataset = TensorDataset(x, z, c_onehot)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

encoder_ckpt_dir = "checkpoints/encoder_concat"
os.makedirs(encoder_ckpt_dir, exist_ok=True)

best_loss = float('inf')

for epoch in tqdm(range(60), desc="Training Epochs"):
    E.train()
    total_loss = 0

    for x_batch, z_batch, c_batch in loader:
        x_batch, z_batch, c_batch = x_batch.to(device), z_batch.to(device), c_batch.to(device)

        z_pred = E(x_batch, c_batch)
        loss = criterion(z_pred, z_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x_batch.size(0)

    avg_loss = total_loss / len(dataset)
    print(f"Epoch {epoch+1:02d} | Loss: {avg_loss:.6f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        best_path = os.path.join(encoder_ckpt_dir, "E_best.pth")
        torch.save(E.state_dict(), best_path)
        print(f"📌 Best model updated at epoch {epoch+1:02d} (Loss: {avg_loss:.6f})")

last_path = os.path.join(encoder_ckpt_dir, "E_last.pth")
torch.save(E.state_dict(), last_path)
print(f"✅ Last model saved: {last_path}")

In [ ]:
import os
import random
from PIL import Image
import torch
from torchvision import transforms

# -------------------------------
# -------------------------------
afac_txt_path = "AFAC/AFAC.txt"
image_root = "AFAC/trainset"
n_samples = 10
img_size = 128
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# -------------------------------
# -------------------------------
transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),                 # [0, 1]
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),  # [-1, 1]
])

# -------------------------------
# -------------------------------
with open(afac_txt_path, "r") as f:
    lines = f.readlines()

valid_lines = [line.strip() for line in lines if "Young" in line or "Senior" in line]

selected = random.sample(valid_lines, n_samples)

# -------------------------------
# -------------------------------
images = []
labels = []

for line in selected:
    filename, label = line.split("\t")
    path = os.path.join(image_root, filename)

    img = Image.open(path).convert("RGB")
    img = transform(img)
    images.append(img)

    label_tensor = torch.tensor([1, 0]) if label == "Young" else torch.tensor([0, 1])
    labels.append(label_tensor)

# -------------------------------
# 3. Stack
# -------------------------------
x_target = torch.stack(images).to(device)  # [5, 3, 128, 128]
c_target = torch.stack(labels).float().to(device)  # [5, 2]

print(f"✅ 총 {n_samples}개 샘플 로딩 및 전처리 완료")
print(f"x_target.shape: {x_target.shape}")
print(f"c_target.shape: {c_target.shape}")

In [ ]:
# Extract VGG features for the perceptual reconstruction loss.

import torchvision.models as models
import torchvision.transforms as transforms

class VGGFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(pretrained=True).features
        self.slice = nn.Sequential(*list(vgg.children())[:36])
        for param in self.slice.parameters():
            param.requires_grad = False

    def forward(self, x):
        return self.slice(x)

perceptual_criterion = nn.L1Loss()
vgg = VGGFeatureExtractor().to(device)

In [ ]:
import torch
import torch.nn as nn

class TotalVariationLoss(nn.Module):
    """
    Total Variation Loss: 이미지의 노이즈를 줄이고 더 자연스럽게 만들기 위해
    인접한 픽셀 간의 차이를 최소화합니다.
    """
    def __init__(self):
        super(TotalVariationLoss, self).__init__()

    def forward(self, img):
        loss_h = torch.mean(torch.abs(img[:, :, 1:, :] - img[:, :, :-1, :]))
        loss_w = torch.mean(torch.abs(img[:, :, :, 1:] - img[:, :, :, :-1]))
        return loss_h + loss_w

In [ ]:
# Optimize latent vectors to reconstruct images before changing the age condition.

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.utils import save_image
from tqdm import tqdm
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF

def tensor_to_pil(img_tensor):
    img_tensor = img_tensor.detach().cpu()
    img_tensor = (img_tensor + 1) / 2
    return TF.to_pil_image(img_tensor.squeeze(0))

# -------------------------------
# -------------------------------
z_dim = 128
c_dim = 2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
save_dir = "results/latent_opt_concat"
os.makedirs(save_dir, exist_ok=True)

# -------------------------------
# -------------------------------
G = InjectionGenerator(z_dim=z_dim, c_dim=c_dim).to(device)
G.load_state_dict(torch.load("checkpoints/injected/G_epoch_250.pth")['G_state_dict'])
G.eval()

# E = Encoder(z_dim=z_dim, c_dim=c_dim).to(device)
E = ConcatEncoder(z_dim=z_dim, c_dim=c_dim).to(device)
# E.load_state_dict(torch.load("checkpoints/encoder_baseline/E_best.pth"))
E.load_state_dict(torch.load("checkpoints/encoder_concat/E_best.pth"))
E.eval()

# -------------------------------
# -------------------------------
n_steps = 1000
lr = 0.01
criterion = nn.MSELoss()

# -------------------------------
# -------------------------------
for i in range(10):
    print(f"\n🐶 Optimizing image {i}...")

    x_i = x_target[i].unsqueeze(0).to(device)        # [1, 3, 128, 128]
    c_i = c_target[i].unsqueeze(0).to(device)        # [1, 2]
    c_i_inv = 1 - c_i

    with torch.no_grad():
        z_0 = E(x_i, c_i)                            # [1, 128]

    z_opt = z_0.clone().detach().requires_grad_(True)
    optimizer = optim.Adam([z_opt], lr=lr)
    
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
    
    best_loss = float('inf')
    z_best = None
    
    for step in range(n_steps):
        optimizer.zero_grad()
        
        x_recon = G(z_opt, c_i)
    
        # 1. Pixel-wise loss
        pixel_loss = criterion(x_recon, x_i)
    
        # 2. Perceptual loss
        x_i_vgg = normalize(x_i.squeeze(0)).unsqueeze(0)       # [1, 3, 128, 128]
        x_recon_vgg = normalize(x_recon.squeeze(0)).unsqueeze(0)
    
        feat_i = vgg(x_i_vgg)
        feat_recon = vgg(x_recon_vgg)
    
        perceptual_loss = perceptual_criterion(feat_i, feat_recon)
    
        loss = pixel_loss + 0.05 * perceptual_loss
    
        loss.backward()
        optimizer.step()
    
        loss_val = loss.item()
        if loss_val < best_loss:
            best_loss = loss_val
            z_best = z_opt.detach().clone()
    
        if step % 100 == 0 or step == n_steps - 1:
            print(f"Step {step+1:4d}/{n_steps} | Total Loss: {loss_val:.6f} | Pixel: {pixel_loss.item():.6f} | Perceptual: {perceptual_loss.item():.6f}")

    with torch.no_grad():
        x_recon_best = G(z_best, c_i)
        x_transformed = G(z_best, c_i_inv)

    your_label = torch.argmax(c_i).item()
    titles = [
        "Original",
        f"Reconstructed ({'Young' if your_label == 0 else 'Senior'})",
        f"Transformed ({'Senior' if your_label == 0 else 'Young'})"
    ]
    
    imgs = [x_i, x_recon_best, x_transformed]
    pil_imgs = [tensor_to_pil(img) for img in imgs]
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, img, title in zip(axes, pil_imgs, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=12)
        ax.axis('off')
    
    plt.tight_layout()
    save_path = os.path.join(save_dir, f"dog_{i}_labeled.png")
    plt.savefig(save_path)
    plt.close()
    print(f"✅ 시각화 저장 완료: {save_path}")

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.utils import save_image
from PIL import Image
from tqdm import tqdm

# -------------------------------
# -------------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
z_dim = 128
c_dim = 2
img_size = 128
n_steps = 1000
lr = 0.01
criterion = nn.MSELoss()

save_root = "results/output"
os.makedirs(save_root, exist_ok=True)
save_dirs = {
    "original": os.path.join(save_root, "original"),
    "recon": os.path.join(save_root, "recon"),
    "transformed": os.path.join(save_root, "transformed")
}
for d in save_dirs.values():
    os.makedirs(d, exist_ok=True)

# -------------------------------
# -------------------------------
G = InjectionGenerator(z_dim=z_dim, c_dim=c_dim).to(device)
G.load_state_dict(torch.load("checkpoints/injected/G_epoch_250.pth")["G_state_dict"])
G.eval()

E = ConcatEncoder(z_dim=z_dim, c_dim=c_dim).to(device)
E.load_state_dict(torch.load("checkpoints/encoder_concat/E_best.pth"))
E.eval()

# 🔹 VGG Perceptual Loss
vgg = VGGFeatureExtractor().to(device)
vgg.eval()
perceptual_criterion = nn.L1Loss()

# 🔹 Normalization (for VGG)
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

# 🔹 Transform (for input image)
transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# 🔹 Denormalization (for saving)
def denorm(t): return (t * 0.5 + 0.5).clamp(0, 1)

# -------------------------------
# -------------------------------
img_dir = "dogs"
img_paths = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('jpg', 'jpeg', 'png'))])[:5]

for fname in tqdm(img_paths, desc="Latent Optimization"):
    path = os.path.join(img_dir, fname)
    img = Image.open(path).convert("RGB")
    x_i = transform(img).unsqueeze(0).to(device)

    if "young" in fname.lower():
        c_i = torch.tensor([[1.0, 0.0]], device=device)
    elif "senior" in fname.lower():
        c_i = torch.tensor([[0.0, 1.0]], device=device)
    else:
        raise ValueError(f"Cannot infer label from filename: {fname}")
    c_i_inv = 1 - c_i

    with torch.no_grad():
        z_0 = E(x_i, c_i)

    z_opt = z_0.clone().detach().requires_grad_(True)
    optimizer = optim.Adam([z_opt], lr=lr)

    best_loss = float('inf')
    z_best = None

    for step in range(n_steps):
        optimizer.zero_grad()
        x_recon = G(z_opt, c_i)

        # Pixel-wise
        pixel_loss = criterion(x_recon, x_i)

        # Perceptual
        x_i_vgg = normalize(x_i.squeeze(0)).unsqueeze(0)
        x_recon_vgg = normalize(x_recon.squeeze(0)).unsqueeze(0)
        feat_i = vgg(x_i_vgg)
        feat_recon = vgg(x_recon_vgg)
        perceptual_loss = perceptual_criterion(feat_i, feat_recon)

        # Total loss
        loss = pixel_loss + 0.05 * perceptual_loss
        loss.backward()
        optimizer.step()

        if loss.item() < best_loss:
            best_loss = loss.item()
            z_best = z_opt.detach().clone()

        if step % 200 == 0 or step == n_steps - 1:
            print(f"[{fname}] Step {step:4d} | Loss: {loss.item():.4f}")

    with torch.no_grad():
        x_recon_best = G(z_best, c_i)
        x_trans = G(z_best, c_i_inv)

    base = os.path.splitext(fname)[0]
    save_image(denorm(x_i), os.path.join(save_dirs["original"], f"{base}.png"))
    save_image(denorm(x_recon_best), os.path.join(save_dirs["recon"], f"{base}.png"))
    save_image(denorm(x_trans), os.path.join(save_dirs["transformed"], f"{base}.png"))